# ⚽ World Cup Intelligence Platform  
## Match Prediction Model

This notebook builds a machine learning model to predict football match outcomes using team ranking differences and historical results.

In [1]:
import pandas as pd
import numpy as np
import random
from collections import defaultdict
from itertools import combinations 
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split


## 📂 Load Dataset
We use a simple dataset containing historical match results and team rankings.

In [2]:
df = pd.read_csv("../data/matches.csv")

df.head()

,team1,team2,team1_goals,team2_goals,team1_rank,team2_rank
0,Brazil,Argentina,2,1,1,2
1,France,Germany,3,2,2,6
2,Spain,Portugal,1,1,7,5
3,England,Italy,2,0,4,8
4,Morocco,Spain,1,0,11,7


## 🧠 Feature Engineering
We convert match results into a supervised learning format:

- 1 → Team 1 wins  
- 0 → Draw  
- -1 → Team 2 wins

In [3]:
df = df.copy()

df["result"] = df.apply(
    lambda x: 1 if x["team1_goals"] > x["team2_goals"]
    else (0 if x["team1_goals"] == x["team2_goals"] else -1),
    axis=1
)

X = df[["team1_rank", "team2_rank"]]
y = df["result"]

X.head(), y.head()

(   team1_rank  team2_rank
 0           1           2
 1           2           6
 2           7           5
 3           4           8
 4          11           7,
 0    1
 1    1
 2    0
 3    1
 4    1
 Name: result, dtype: int64)

## ✂️ Train/Test Split
We split the dataset to evaluate model performance.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## 🤖 Train Model
We use a Random Forest Classifier.

In [5]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstr

## 📊 Model Prediction
We test predictions using team rankings.

In [6]:
sample = [[1, 10]]  # Strong team vs weak team

prediction = model.predict(sample)
probability = model.predict_proba(sample)

prediction, probability

c:\Users\User\OneDrive\Desktop\moringa\Other projects\world-cup-intelligence\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\User\OneDrive\Desktop\moringa\Other projects\world-cup-intelligence\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


(array([1]), array([[0.04, 0.27, 0.69]]))

## 📌 Interpretation
- Prediction shows match outcome class
- Probability shows confidence for each class:
  - [-1] Team 2 win
  - [0] Draw
  - [1] Team 1 win

In [7]:
sample = pd.DataFrame([[1, 10]], columns=["team1_rank", "team2_rank"])
prediction = model.predict(sample)
probability = model.predict_proba(sample)

prediction, probability

(array([1]), array([[0.04, 0.27, 0.69]]))

## 🧠 Build Team Strength Model

We estimate team strength using average ranking performance.
Lower rank = stronger team.

In [8]:
teams = pd.concat([df["team1"], df["team2"]]).unique()

team_strength = {}

for team in teams:
    ranks = []

    if team in df["team1"].values:
        ranks.extend(df[df["team1"] == team]["team1_rank"].tolist())

    if team in df["team2"].values:
        ranks.extend(df[df["team2"] == team]["team2_rank"].tolist())

    team_strength[team] = np.mean(ranks)

team_strength

{'Brazil': np.float64(1.0),
 'France': np.float64(2.0),
 'Spain': np.float64(7.0),
 'England': np.float64(4.0),
 'Morocco': np.float64(11.0),
 'Japan': np.float64(18.0),
 'USA': np.float64(15.0),
 'Argentina': np.float64(2.0),
 'Germany': np.float64(6.0),
 'Portugal': np.float64(5.0),
 'Italy': np.float64(8.0),
 'Croatia': np.float64(12.0),
 'Mexico': np.float64(13.0)}

## ⚔️ Match Simulation Function

We convert team strength into win probability.

In [9]:
def simulate_match(team1, team2):
    # .get(..., 20) prevents the KeyError by giving placeholder teams a rating of 20
    r1 = team_strength.get(team1, 20)
    r2 = team_strength.get(team2, 20)

    prob_team1 = 1 / (1 + np.exp((r1 - r2) / 5))
    prob_team2 = 1 - prob_team1

    return random.choices(
        [team1, team2],
        weights=[prob_team1, prob_team2]
    )[0]


## 🏆 Knockout Simulation

We simulate a simple knockout bracket (64 → 1 winner logic simplified for MVP).

In [10]:
def run_tournament(teams):
    round_teams = teams.copy()

    while len(round_teams) > 1:
        next_round = []

        random.shuffle(round_teams)

        for i in range(0, len(round_teams), 2):
            if i + 1 < len(round_teams):
                winner = simulate_match(round_teams[i], round_teams[i+1])
                next_round.append(winner)

        round_teams = next_round

    return round_teams[0]

## 🔁 Monte Carlo Simulation

We run the tournament many times to estimate win probabilities.

In [11]:
def monte_carlo_simulation(n_simulations=1000):
    results = defaultdict(int)

    team_list = list(team_strength.keys())

    for _ in range(n_simulations):
        winner = run_tournament(team_list)
        results[winner] += 1

    return results

In [12]:
results = monte_carlo_simulation(500)

results

defaultdict(int,
            {'England': 65,
             'Italy': 24,
             'Portugal': 54,
             'France': 81,
             'Brazil': 113,
             'Argentina': 94,
             'Germany': 41,
             'Spain': 23,
             'Morocco': 2,
             'Mexico': 2,
             'USA': 1})

In [13]:
total = sum(results.values())

win_probabilities = {
    team: round(count / total, 4)
    for team, count in results.items()
}

sorted(win_probabilities.items(), key=lambda x: x[1], reverse=True)[:10]

[('Brazil', 0.226),
 ('Argentina', 0.188),
 ('France', 0.162),
 ('England', 0.13),
 ('Portugal', 0.108),
 ('Germany', 0.082),
 ('Italy', 0.048),
 ('Spain', 0.046),
 ('Morocco', 0.004),
 ('Mexico', 0.004)]

## 📊 Interpretation

This shows:
- Estimated World Cup winner probabilities
- Based on repeated simulations
- Using team strength approximation from historical data

## 🌍 Step 1: Create 48 Teams

We simulate 48 teams (since real qualifiers are not finalized yet).

In [14]:
fortyteams = list(team_strength.keys())

# If less than 48, duplicate + modify (for simulation purposes)
while len(fortyteams) < 48:
    fortyteams.append(f"Team_{len(fortyteams)+1}")

random.shuffle(fortyteams)

len(fortyteams), fortyteams[:10]

(48,
 ['Team_14',
  'Team_45',
  'Brazil',
  'England',
  'USA',
  'Team_17',
  'Team_33',
  'Team_46',
  'Team_37',
  'Team_41'])

## 🧩 Step 2: Create 12 Groups (A–L)
Each group has 4 teams.

In [15]:
group_names = [chr(i) for i in range(65, 77)]  # A–L

groups = {}

for i, group in enumerate(group_names):
    groups[group] = fortyteams[i*4:(i+1)*4]

groups

{'A': ['Team_14', 'Team_45', 'Brazil', 'England'],
 'B': ['USA', 'Team_17', 'Team_33', 'Team_46'],
 'C': ['Team_37', 'Team_41', 'Team_34', 'Team_39'],
 'D': ['Argentina', 'Team_22', 'Team_40', 'Japan'],
 'E': ['Team_44', 'Croatia', 'Team_31', 'Team_38'],
 'F': ['Team_43', 'France', 'Team_32', 'Italy'],
 'G': ['Team_24', 'Team_21', 'Team_27', 'Team_15'],
 'H': ['Team_16', 'Team_28', 'Team_29', 'Team_23'],
 'I': ['Team_20', 'Spain', 'Team_42', 'Mexico'],
 'J': ['Team_25', 'Germany', 'Team_19', 'Team_30'],
 'K': ['Team_47', 'Team_26', 'Team_48', 'Team_36'],
 'L': ['Portugal', 'Morocco', 'Team_18', 'Team_35']}

## ⚔️ Step 3: Match Simulation (Group Stage)
Each team plays every other team in the group.

In [16]:
def simulate_group_match(team1, team2):
    r1 = team_strength.get(team1, 20)
    r2 = team_strength.get(team2, 20)

    prob1 = 1 / (1 + np.exp((r1 - r2) / 5))
    prob2 = 1 - prob1

    result = random.choices(
        ["team1", "team2", "draw"],
        weights=[prob1, prob2, 0.2]
    )[0]

    return result

## 📊 Step 4: Group Standings Calculation
We track points for each team.

In [17]:
def play_group(group_teams):
    standings = defaultdict(int)

    matches = list(combinations(group_teams, 2))

    for t1, t2 in matches:
        result = simulate_group_match(t1, t2)

        if result == "team1":
            standings[t1] += 3
        elif result == "team2":
            standings[t2] += 3
        else:
            standings[t1] += 1
            standings[t2] += 1

    return standings

## 🏆 Step 5: Determine Qualified Teams
Top 2 from each group qualify.

In [18]:
qualified = []

group_results = {}

for group, group_teams in groups.items():
    standings = play_group(group_teams)

    sorted_teams = sorted(
        standings.items(),
        key=lambda x: x[1],
        reverse=True
    )

    top2 = [sorted_teams[0][0], sorted_teams[1][0]]

    qualified.extend(top2)
    group_results[group] = sorted_teams

qualified, len(qualified)

(['Brazil',
  'England',
  'USA',
  'Team_33',
  'Team_41',
  'Team_37',
  'Argentina',
  'Team_40',
  'Croatia',
  'Team_44',
  'France',
  'Italy',
  'Team_15',
  'Team_24',
  'Team_28',
  'Team_29',
  'Spain',
  'Mexico',
  'Germany',
  'Team_25',
  'Team_26',
  'Team_48',
  'Portugal',
  'Morocco'],
 24)

## ⚔️ Step 6: Knockout Stage (Round of 32)
We now simulate elimination rounds.

In [19]:
def simulate_knockout(teams_list):
    round_teams = teams_list.copy()

    while len(round_teams) > 1:
        next_round = []

        random.shuffle(round_teams)

        for i in range(0, len(round_teams), 2):
            if i + 1 < len(round_teams):
                winner = simulate_match(round_teams[i], round_teams[i+1])
                next_round.append(winner)

        round_teams = next_round

    return round_teams[0]

In [20]:
winner = simulate_knockout(qualified)

winner

'Argentina'

## 📊 Monte Carlo Simulation (Full Tournament)

We repeat the full World Cup simulation many times.

In [21]:
def full_world_cup_simulation(n=200):
    results = defaultdict(int)

    for _ in range(n):
        random.shuffle(qualified)
        winner = simulate_knockout(qualified)
        results[winner] += 1

    return results

In [22]:
results = full_world_cup_simulation(200)

total = sum(results.values())

win_probs = {
    team: round(count / total, 4)
    for team, count in results.items()
}

sorted(win_probs.items(), key=lambda x: x[1], reverse=True)[:10]

[('Argentina', 0.21),
 ('Brazil', 0.205),
 ('France', 0.155),
 ('England', 0.135),
 ('Portugal', 0.085),
 ('Germany', 0.08),
 ('Spain', 0.06),
 ('Croatia', 0.025),
 ('Italy', 0.025),
 ('Morocco', 0.015)]

# 📓 NOTEBOOK SECTION — “REALISTIC MATCH ENGINE”

# ⚽ Real Match Scoring Engine

We now simulate real football outcomes:
- Goals scored per team
- Draws
- Extra randomness
- Strength-based scoring
- Penalty shootouts (knockout stage)

This makes the simulator much closer to real World Cup behavior.

## 🧠 Step 1: Convert Team Strength → Attack Power

We transform ranking into expected goals.
Lower rank = stronger team = more goals.

In [23]:
def expected_goals(rank):
    return max(0.3, 3.5 - (rank / 10))

## ⚽ Step 2: Simulate Match Goals

We use Poisson distribution (standard in football analytics).

In [24]:
def simulate_goals(team1_rank, team2_rank):
    g1 = np.random.poisson(expected_goals(team1_rank))
    g2 = np.random.poisson(expected_goals(team2_rank))
    return g1, g2

## ⚔️ Step 3: Real Match Result Function

Now we simulate:
- Goals
- Draws
- Winner determination

In [25]:
def simulate_real_match(team1, team2, knockout=False):
    r1 = team_strength.get(team1, 20)
    r2 = team_strength.get(team2, 20)

    g1, g2 = simulate_goals(r1, r2)

    if g1 > g2:
        return team1, g1, g2

    elif g2 > g1:
        return team2, g1, g2

    else:
        # DRAW CASE
        if knockout:
            # Penalty shootout
            winner = random.choice([team1, team2])
            return winner, g1, g2

        return "draw", g1, g2

## 📊 Step 4: Update Group Match Logic

Now we include:
- Goals
- Goal difference
- Points system

In [26]:
def simulate_group_match_real(t1, t2):
    result, g1, g2 = simulate_real_match(t1, t2)

    return result, g1, g2

In [27]:
def play_group_real(group_teams):
    table = {team: {"pts": 0, "gf": 0, "ga": 0} for team in group_teams}

    from itertools import combinations

    for t1, t2 in combinations(group_teams, 2):
        result, g1, g2 = simulate_group_match_real(t1, t2)

        table[t1]["gf"] += g1
        table[t1]["ga"] += g2

        table[t2]["gf"] += g2
        table[t2]["ga"] += g1

        if result == "draw":
            table[t1]["pts"] += 1
            table[t2]["pts"] += 1
        elif result == t1:
            table[t1]["pts"] += 3
        else:
            table[t2]["pts"] += 3

    return table

## 🏆 Step 5: Ranking Teams in Group

We now use:
1. Points
2. Goal difference
3. Goals scored

In [28]:
def rank_group(table):
    return sorted(
        table.items(),
        key=lambda x: (
            x[1]["pts"],
            x[1]["gf"] - x[1]["ga"],
            x[1]["gf"]
        ),
        reverse=True
    )

## ⚽ Step 6: Apply to Full Groups

In [29]:
qualified = []

for group, teams in groups.items():
    table = play_group_real(teams)
    ranked = rank_group(table)

    top2 = [ranked[0][0], ranked[1][0]]
    qualified.extend(top2)

len(qualified), qualified[:10]

(24,
 ['Brazil',
  'England',
  'USA',
  'Team_46',
  'Team_34',
  'Team_39',
  'Team_22',
  'Team_40',
  'Croatia',
  'Team_44'])

## ⚔️ Step 7: Update Knockout to Use Real Matches

In [30]:
def knockout_real(teams_list):
    round_teams = teams_list.copy()

    while len(round_teams) > 1:
        next_round = []
        random.shuffle(round_teams)

        for i in range(0, len(round_teams), 2):
            if i + 1 < len(round_teams):
                winner, _, _ = simulate_real_match(
                    round_teams[i],
                    round_teams[i+1],
                    knockout=True
                )
                next_round.append(winner)

        round_teams = next_round

    return round_teams[0]